In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('D:/downloads/woc\'26/Checkpoint_2/ml/data/placementdata.csv')

In [3]:
df

,StudentID,CGPA,Internships,Projects,Workshops/Certifications,AptitudeTestScore,SoftSkillsRating,ExtracurricularActivities,PlacementTraining,SSC_Marks,HSC_Marks,PlacementStatus
0,1,7.5,1,1,1,65,4.4,No,No,61,79,NotPlaced
1,2,8.9,0,3,2,90,4.0,Yes,Yes,78,82,Placed
2,3,7.3,1,2,2,82,4.8,Yes,No,79,80,NotPlaced
3,4,7.5,1,1,2,85,4.4,Yes,Yes,81,80,Placed
4,5,8.3,1,2,2,86,4.5,Yes,Yes,74,88,Placed
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,7.5,1,1,2,72,3.9,Yes,No,85,66,NotPlaced
9996,9997,7.4,0,1,0,90,4.8,No,No,84,67,Placed
9997,9998,8.4,1,3,0,70,4.8,Yes,Yes,79,81,Placed
9998,9999,8.9,0,3,2,87,4.8,Yes,Yes,71,85,Placed


In [4]:
df['ExtracurricularActivities'] = df['ExtracurricularActivities'].map(lambda x: 1 if x == 'Yes' else 0)
df['PlacementStatus'] = df['PlacementStatus'].map(lambda x: 1 if x == 'Placed' else 0)
df['PlacementTraining'] = df['PlacementTraining'].map(lambda x: 1 if x == 'Yes' else 0)
df

,StudentID,CGPA,Internships,Projects,Workshops/Certifications,AptitudeTestScore,SoftSkillsRating,ExtracurricularActivities,PlacementTraining,SSC_Marks,HSC_Marks,PlacementStatus
0,1,7.5,1,1,1,65,4.4,0,0,61,79,0
1,2,8.9,0,3,2,90,4.0,1,1,78,82,1
2,3,7.3,1,2,2,82,4.8,1,0,79,80,0
3,4,7.5,1,1,2,85,4.4,1,1,81,80,1
4,5,8.3,1,2,2,86,4.5,1,1,74,88,1
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,7.5,1,1,2,72,3.9,1,0,85,66,0
9996,9997,7.4,0,1,0,90,4.8,0,0,84,67,1
9997,9998,8.4,1,3,0,70,4.8,1,1,79,81,1
9998,9999,8.9,0,3,2,87,4.8,1,1,71,85,1


In [5]:
def create_new_features(df):
    df_eng = df.copy()
    
    # 1. ACADEMIC PERFORMANCE COMPOSITE
    # Combine different academic metrics
    if all(col in df.columns for col in ['CGPA', 'SSC_Marks', 'HSC_Marks']):
        # Normalize each academic metric
        df_eng['Academic_Normalized'] = (
            (df['CGPA'] - df['CGPA'].min()) / (df['CGPA'].max() - df['CGPA'].min()) * 0.5 +
            (df['SSC_Marks'] - df['SSC_Marks'].min()) / (df['SSC_Marks'].max() - df['SSC_Marks'].min()) * 0.2 +
            (df['HSC_Marks'] - df['HSC_Marks'].min()) / (df['HSC_Marks'].max() - df['HSC_Marks'].min()) * 0.3
        )
        df_eng['Academic_Composite'] = (df_eng['Academic_Normalized'] * 100).round(1)
        print("Created Academic_Composite")
    
    # 2. PRACTICAL EXPERIENCE SCORE
    # Combine internships, projects, workshops
    practical_score = 0
    weight_sum = 0
    
    if 'Internships' in df.columns:
        practical_score += df['Internships'] * 0.5
        weight_sum += 0.4
    if 'Projects' in df.columns:
        practical_score += df['Projects'] * 0.3
        weight_sum += 0.3
    if 'Workshops/Certifications' in df.columns:
        practical_score += df['Workshops/Certifications'] * 0.2
        weight_sum += 0.2
    
    if weight_sum > 0:
        df_eng['Practical_Experience_Score'] = (practical_score / weight_sum).round(2)
        print(" Created Practical_Experience_Score")
    
    # 3. SKILL DEVELOPMENT INDEX
    if all(col in df.columns for col in ['AptitudeTestScore', 'SoftSkillsRating']):
        df_eng['Skill_Development_Index'] = (
            df['AptitudeTestScore'] * 0.6 + 
            df['SoftSkillsRating'] * 0.4
        )
        df_eng['Skill_Development_Index'] = (
            (df_eng['Skill_Development_Index'] - df_eng['Skill_Development_Index'].min()) /
            (df_eng['Skill_Development_Index'].max() - df_eng['Skill_Development_Index'].min()) * 100
        ).round(1)
        print("Created Skill_Development_Index")
    
    # 4. OVERALL ENGAGEMENT SCORE
    engagement_features = []
    if 'ExtracurricularActivities' in df.columns:
        engagement_features.append('ExtracurricularActivities')
    if 'PlacementTraining' in df.columns:
        engagement_features.append('PlacementTraining')
    if 'Workshops/Certifications' in df.columns:
        engagement_features.append('Workshops/Certifications')
    
    if engagement_features:
        df_eng['Engagement_Score'] = df[engagement_features].sum(axis=1)
        print("Created Engagement_Score")
    
    # 7. COMPETITIVENESS SCORE (placement-specific)
    important_features = ['CGPA', 'AptitudeTestScore', 'Internships', 'Projects']
    available_features = [f for f in important_features if f in df.columns]
    
    if len(available_features) >= 2:
        # Normalize each feature
        normalized = pd.DataFrame()
        for feature in available_features:
            normalized[f'{feature}_norm'] = (
                (df[feature] - df[feature].min()) / 
                (df[feature].max() - df[feature].min())
            )
        
        # Weighted average (CGPA and Aptitude are most important)
        weights = {'CGPA': 0.4, 'AptitudeTestScore': 0.3, 'Internships': 0.15, 'Projects': 0.15}
        
        competitiveness_score = 0
        for feature in available_features:
            if f'{feature}_norm' in normalized.columns:
                competitiveness_score += normalized[f'{feature}_norm'] * weights.get(feature, 0.1)
        
        df_eng['Competitiveness_Score'] = (competitiveness_score * 100).round(1)
        print(f" Created Competitiveness_Score using {available_features}")    
    return df_eng

In [6]:
df_engineered = create_new_features(df)
df_engineered.head()

Created Academic_Composite
 Created Practical_Experience_Score
Created Skill_Development_Index
Created Engagement_Score
 Created Competitiveness_Score using ['CGPA', 'AptitudeTestScore', 'Internships', 'Projects']


,StudentID,CGPA,Internships,Projects,Workshops/Certifications,AptitudeTestScore,SoftSkillsRating,ExtracurricularActivities,PlacementTraining,SSC_Marks,HSC_Marks,PlacementStatus,Academic_Normalized,Academic_Composite,Practical_Experience_Score,Skill_Development_Index,Engagement_Score,Competitiveness_Score
0,1,7.5,1,1,1,65,4.4,0,0,61,79,0,0.439497,43.9,1.11,19.0,1,32.9
1,2,8.9,0,3,2,90,4.0,1,1,78,82,1,0.834903,83.5,1.44,98.3,4,81.9
2,3,7.3,1,2,2,82,4.8,1,0,79,80,0,0.513570,51.4,1.67,74.4,3,51.8
3,4,7.5,1,1,2,85,4.4,1,1,81,80,1,0.563460,56.3,1.33,83.1,4,52.9
4,5,8.3,1,2,2,86,4.5,1,1,74,88,1,0.754725,75.5,1.67,86.5,4,71.2


In [7]:
from sklearn.feature_selection import SelectKBest,f_classif
def perform_feature_selection(df, target='PlacementStatus'):                                                                  

    X = df.drop(columns=[target] if target in df.columns else [])
    y = df[target] if target in df.columns else None
    
    print(f"Starting feature selection on {X.shape[1]} numeric features...")

    selection_results = {}
    
    # Correlation with target
    if y is not None:
        correlations = X.apply(lambda col: col.corr(y))
        selection_results['Correlation'] = correlations.abs().sort_values(ascending=False)
    
    if y is not None:
        selector_anova = SelectKBest(score_func=f_classif, k='all')
        selector_anova.fit(X, y)
        selection_results['ANOVA_F_Score'] = pd.Series(
            selector_anova.scores_, 
            index=X.columns
        ).sort_values(ascending=False)
    
    return selection_results, X.columns

selection_results, numeric_features = perform_feature_selection(df_engineered)

for method, scores in selection_results.items():
    print(f"\n-- {method} - Top 10 Features:")
    print(scores.head(10))

Starting feature selection on 17 numeric features...

-- Correlation - Top 10 Features:
Competitiveness_Score         0.572341
Academic_Composite            0.568812
Academic_Normalized           0.568811
Skill_Development_Index       0.526790
AptitudeTestScore             0.521744
Engagement_Score              0.517252
HSC_Marks                     0.505746
ExtracurricularActivities     0.482491
Practical_Experience_Score    0.477586
Projects                      0.475186
dtype: float64

-- ANOVA_F_Score - Top 10 Features:
Competitiveness_Score         4870.562842
Academic_Composite            4782.046269
Academic_Normalized           4782.004089
Skill_Development_Index       3840.206221
AptitudeTestScore             3739.602478
Engagement_Score              3652.072731
HSC_Marks                     3436.189189
ExtracurricularActivities     3033.766774
Practical_Experience_Score    2954.266330
Projects                      2916.004327
dtype: float64


In [8]:
def get_consensus_features(selection_results, top_k=7):
    consensus_scores = pd.DataFrame(
        index=selection_results[next(iter(selection_results))].index
    )

    for method, scores in selection_results.items():
        scores = scores.abs()
        normalized = (scores - scores.min()) / (scores.max() - scores.min())
        consensus_scores[method] = normalized

    consensus_scores['Average_Score'] = consensus_scores.mean(axis=1)
    consensus_scores = consensus_scores.sort_values(
        'Average_Score', ascending=False
    )

    return consensus_scores.head(top_k).index.tolist(), consensus_scores
top_features, consensus_df = get_consensus_features(selection_results, top_k=7)

top_features

['Competitiveness_Score',
 'Academic_Composite',
 'Academic_Normalized',
 'Skill_Development_Index',
 'AptitudeTestScore',
 'Engagement_Score',
 'HSC_Marks']

In [9]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

X = df_engineered[top_features]
y = df_engineered['PlacementStatus']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
)

rf_param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [5, 8, 12, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_search = RandomizedSearchCV(
    rf,
    rf_param_grid,
    n_iter=30,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X, y)

print("Best RF Params:", rf_search.best_params_)
print("Best RF ROC-AUC:", rf_search.best_score_)


Best RF Params: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 5}
Best RF ROC-AUC: 0.866853626930159


In [11]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    eval_metric='logloss',
    random_state=42
)

xgb_param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [3, 4, 6],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.2]
}

xgb_search = RandomizedSearchCV(
    xgb,
    xgb_param_grid,
    n_iter=30,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X, y)

print("Best XGB Params:", xgb_search.best_params_)
print("Best XGB ROC-AUC:", xgb_search.best_score_)


Best XGB Params: {'subsample': 0.9, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 0.7}
Best XGB ROC-AUC: 0.868898015086053


In [12]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    random_state=42,
    verbose=-1,  # Already set
    force_col_wise=True  # Addresses the threading warning too
)

lgbm_param_grid = {
    'n_estimators': [200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'num_leaves': [15, 31, 63],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

lgbm_search = RandomizedSearchCV(
    lgbm,
    lgbm_param_grid,
    n_iter=30,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1
)

lgbm_search.fit(X, y)

print("Best LGBM Params:", lgbm_search.best_params_)
print("Best LGBM ROC-AUC:", lgbm_search.best_score_)


Best LGBM Params: {'subsample': 0.9, 'num_leaves': 31, 'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.8}
Best LGBM ROC-AUC: 0.8689751891103026


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib


X = df_engineered[top_features]
y = df_engineered['PlacementStatus']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

models = {
    'XGBoost': XGBClassifier(
        **xgb_search.best_params_,
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=42
    ),
    'LightGBM': LGBMClassifier(
        **lgbm_search.best_params_,
        random_state=42,
        verbose=-1
    ),
    'RandomForest': RandomForestClassifier(
        **rf_search.best_params_,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )
}


In [14]:
results = {}
best_model = None
best_score = 0

for model_name, model in models.items():
    print(f"\n{'='*40}")
    print(f"Training {model_name}")
    print(f"{'='*40}")
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    f1 = f1_score(y_test, y_pred)
    
    results[model_name] = {
        'model': model,
        'accuracy': accuracy,
        'roc_auc': roc_auc,
        'f1_score': f1
    }
    
    print(f"Accuracy : {accuracy:.3f}")
    print(f"ROC-AUC  : {roc_auc:.3f}")
    print(f"F1-Score : {f1:.3f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Not Placed', 'Placed']))
    
    if roc_auc > best_score:
        best_score = roc_auc
        best_model_name = model_name
        best_model = model



Training XGBoost


d:\downloads\woc'26\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [11:03:29] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy : 0.794
ROC-AUC  : 0.875
F1-Score : 0.747

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.81      0.84      0.83      1161
      Placed       0.77      0.72      0.75       839

    accuracy                           0.79      2000
   macro avg       0.79      0.78      0.79      2000
weighted avg       0.79      0.79      0.79      2000


Training LightGBM
Accuracy : 0.794
ROC-AUC  : 0.875
F1-Score : 0.748

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.81      0.84      0.83      1161
      Placed       0.77      0.73      0.75       839

    accuracy                           0.79      2000
   macro avg       0.79      0.79      0.79      2000
weighted avg       0.79      0.79      0.79      2000


Training RandomForest
Accuracy : 0.792
ROC-AUC  : 0.872
F1-Score : 0.762

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.84 

In [15]:
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)

for model_name, metrics in results.items():
    print(f"\n{model_name}")
    print(f"Accuracy : {metrics['accuracy']:.3f}")
    print(f"ROC-AUC  : {metrics['roc_auc']:.3f}")
    print(f"F1-Score : {metrics['f1_score']:.3f}")

print(f"\n🎯 BEST MODEL: {best_model_name} (ROC-AUC: {best_score:.3f})")

# Save best model
joblib.dump(best_model, f'placement_predictor_best_model_{best_model_name.lower()}.pkl')
print(f"Best model saved as placement_predictor_best_model_{best_model_name.lower()}.pkl")

# # Optional: save all models
for model_name, metrics in results.items():
    joblib.dump(metrics['model'], f'placement_predictor_{model_name.lower()}.pkl')

print("All models saved for comparison")


MODEL COMPARISON SUMMARY

XGBoost
Accuracy : 0.794
ROC-AUC  : 0.875
F1-Score : 0.747

LightGBM
Accuracy : 0.794
ROC-AUC  : 0.875
F1-Score : 0.748

RandomForest
Accuracy : 0.792
ROC-AUC  : 0.872
F1-Score : 0.762

🎯 BEST MODEL: LightGBM (ROC-AUC: 0.875)
Best model saved as placement_predictor_best_model_lightgbm.pkl
All models saved for comparison
